# FLAN-T5 for Figure-Attribute Extraction (Heliophysics)

Text-to-text fine-tuning of FLAN-T5-Large: figure caption in, structured attributes out (`FigureType`, `MeasurementUnit`, `ScientificObservable`).

**Notes**
- Designed to run on a GPU notebook (Kaggle T4). The annotated captions are not included here; set `DATA_FILE` below.
- Test set: 69 captions. FigureType accuracy 72.4% (0 invalid generations), MeasurementUnit set-F1 0.715 (normalized), ScientificObservable set-F1 0.422 (strict exact match).
- The cells below write the two scripts to disk (`%%writefile`), train, then run a seen-vs-unseen vocabulary analysis.


In [1]:
%%writefile train_generative_figure_model.py

"""
Generative model for figure attributes: FigureType, MeasurementUnit, and
ScientificObservable, all three, in one model.

Fine-tunes FLAN-T5 (free, open-weight) as a text-to-text generator:
caption in, structured attribute string out.

This is the GENERATIVE counterpart to the EXTRACTIVE entities model --
together they justify the methodological split: "entity types are literal
text spans (extractive/NER-appropriate); figure attributes often require
inference beyond literal text (generative/seq2seq-appropriate)."

Target format (verified round-trip on real data, 0 mismatches / 672 examples):
    "type: <FigureType> | units: <comma-separated units, or 'none'> | observable: <comma-separated observables, or 'none'>"

Evaluation:
    - FigureType: standard classification accuracy/F1 (closed 5-class set)
    - MeasurementUnit, ScientificObservable: set-overlap F1, reported both
      RAW (exact string match after lowercasing) and NORMALIZED (light
      punctuation/plural cleanup, or a data-driven alias table for units --
      see below) -- raw is punishing on trivial phrasing/formatting
      differences, normalized is a fairer reflection of real quality.

Fixes vs. the original draft:
    - Precision is auto-selected (bf16 on Ampere+, fp16 on older GPUs like
      T4, fp32 on CPU) instead of a hardcoded bf16 flag that silently
      breaks or under-performs on Colab/Kaggle T4s.
    - Dynamic padding (pad per-batch, not to a fixed max_length) -- saves
      real compute since captions vary a lot in length.
    - Checkpoint selection uses a real decoded metric (mean of FigureType
      accuracy + normalized unit F1 + normalized observable F1) instead of
      eval_loss, which can diverge from actual generation quality.
    - --model_size {base,large} with batch size / grad accumulation /
      gradient checkpointing set sensibly for each, so FLAN-T5-large fits
      on a free 16GB T4 without needing LoRA or 8-bit.
    - Class distribution is logged and plotted, since "other" being a
      rare/messy catch-all class is a known risk with this label set.
    - The final saved model directory now also includes an
      inference_config.json (prompt prefix, target parsing/normalization
      rules, generation length) so the model is actually reusable on its
      own later, without needing this training script alongside it.

Tuning specific to this actual dataset (672 real examples, verified by
directly analyzing Heliophysics_Figure_Caption_Annotation.json):
    - FigureType is ~4x imbalanced (time-series=229 vs scatter=57).
      --balance_classes (on by default) oversamples minority classes in
      the TRAIN split only (val/test/held-out folds are never touched, so
      evaluation is never inflated) up to min(majority_count, 2x the
      class's own count).
    - MeasurementUnit is a near-closed vocabulary in this data: only 83
      unique terms across 672 examples, and the same physical unit is
      written multiple inconsistent ways by annotators (e.g. "AU"/"au",
      "degree"/"degrees", "R_sun"/"Rs"/"RS"/"R\u2609"). UNIT_ALIASES below
      is a canonicalization table built directly from the observed terms
      in this file (not guessed) so evaluation isn't punished for
      formatting differences that don't reflect real errors. It also
      fixes a real bug from an earlier draft: a generic cleanup regex
      silently stripped the \u2609 (sun) symbol, turning "R\u2609" into "R"
      -- a wrong merge that actually occurs in this data.
    - ScientificObservable is NOT close to closed vocabulary: 869 unique
      terms across 672 examples, and 645 of those (74%) appear EXACTLY
      ONCE in the whole dataset. This is a hard ceiling, not a tuning
      problem -- a term the model has never seen in training has no way
      to be learned, and a good fraction of held-out terms will fall into
      that bucket. Expect ScientificObservable F1 to be the weakest of
      the three fields, and the vocabulary-richness plot below shows why.

Deliberately NOT included, and why:
    - Data augmentation (back-translation / paraphrasing) was considered
      and rejected for this dataset specifically: 60.7% of MeasurementUnit
      terms and 46.4% of ScientificObservable terms are literal substrings
      of their captions. Generic paraphrasing tools commonly drop or
      reword exactly this kind of literal span (unit symbols, technical
      noun phrases) while the gold label stays fixed, which would silently
      create mismatched training pairs -- more likely to hurt this dataset
      than help it. Class balancing (safe duplication of real examples,
      not synthesis of new text) is the augmentation-like lever used
      instead, since it carries none of that risk.

Usage:
    python train_generative_figure_model.py /path/to/output_folder --data_file Heliophysics_Figure_Caption_Annotation.json
    python train_generative_figure_model.py /path/to/output_folder --data_file <file> --model_size large

    Optional k-fold robustness check (see docstring on run_kfold_cv below
    for what this trades off -- it multiplies total runtime by k):
    python train_generative_figure_model.py /path/to/output_folder --data_file <file> --kfold 5

Reusing a trained model later (see generative_model_final/inference_config.json):
    import json
    from pathlib import Path
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

    model_dir = Path("output_folder/generative_model_final")
    cfg = json.loads((model_dir / "inference_config.json").read_text())
    tok = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)

    caption = "..."
    inputs = tok(cfg["prompt_prefix"] + caption, return_tensors="pt", truncation=True,
                 max_length=cfg["max_input_length"])
    out_ids = model.generate(**inputs, max_length=cfg["max_target_length"])
    raw = tok.decode(out_ids[0], skip_special_tokens=True)
    # then use parse_target(raw) / normalize_unit_term / normalize_obs_term
    # from this script, or reimplement from the rules stored in
    # inference_config.json, to get structured fields + normalized scoring.
"""

import sys
import json
import time
import random
import argparse
import shutil
import re
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    DataCollatorForSeq2Seq, EarlyStoppingCallback,
)
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

FIGURE_TYPES = ["time-series", "spectral", "spatial-map", "scatter", "other"]
SEED = 42
MAX_INPUT_LENGTH = 300
MAX_TARGET_LENGTH = 100
PROMPT_PREFIX = "Extract figure attributes from this caption: "

MODEL_NAMES = {
    "base": "google/flan-t5-base",
    "large": "google/flan-t5-large",
}
# Sensible defaults per model size for a free 16GB T4 (Colab/Kaggle).
# Effective batch size = batch_size * grad_accum in both cases.
DEFAULTS = {
    "base": {"batch_size": 8, "grad_accum": 1, "grad_checkpointing": False},
    "large": {"batch_size": 2, "grad_accum": 8, "grad_checkpointing": True},
}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

sns.set_theme(style="white", context="talk", font_scale=0.72)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
PALETTE = ["#0072B2", "#009E73", "#D55E00", "#CC79A7", "#E69F00", "#56B4E9"]
sns.set_palette(sns.color_palette(PALETTE))
TYPE_COLOR = dict(zip(FIGURE_TYPES, PALETTE))
LEGEND_OUTSIDE = dict(loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False)


def savefig(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved: {path.name}")


# ---------------------------------------------------------------------------
# Target string construction / parsing -- verified via round-trip test on
# real data before this script was written (0 mismatches on 672 examples)
# ---------------------------------------------------------------------------

def build_target(fig_type, unit, obs):
    unit = unit if unit else "none"
    obs = obs if obs else "none"
    return f"type: {fig_type} | units: {unit} | observable: {obs}"


def parse_target(s):
    parts = {}
    for chunk in s.split("|"):
        if ":" in chunk:
            k, v = chunk.split(":", 1)
            parts[k.strip()] = v.strip()
    unit = parts.get("units", "")
    obs = parts.get("observable", "")
    return {
        "type": parts.get("type", ""),
        "unit": "" if unit == "none" else unit,
        "obs": "" if obs == "none" else obs,
    }


# Canonicalization table for MeasurementUnit, built from the actual 83
# unique unit strings observed in Heliophysics_Figure_Caption_Annotation.json
# (see analysis in the module docstring). Only genuinely-equivalent notations
# for the SAME physical unit are merged -- e.g. keV/eV/MeV are kept distinct
# since they differ by orders of magnitude, and R_e (Earth radii) is kept
# distinct from R_sun (solar radii) even though both are "radii".
UNIT_ALIASES = {
    "au": "au",
    "degree": "degree", "degrees": "degree", "deg": "degree", "°": "degree",
    "minutes": "min", "minute": "min", "min": "min",
    "hr": "hr", "hour": "hr", "hours": "hr",
    "r_sun": "r_sun", "rs": "r_sun", "r☉": "r_sun",
    "solar radii": "r_sun", "solar radius": "r_sun",
    "r_e": "r_e", "re": "r_e", "earth radii": "r_e", "earth radius": "r_e",
    "ut": "ut",
    "k": "k", "mk": "mk",  # Kelvin vs megaKelvin -- genuinely different scale, not merged
    "kev": "kev", "ev": "ev", "mev": "mev",  # different scales, not merged
}
# Characters kept as-is for units (superscripts/subscripts and the solar
# symbol carry real meaning here, e.g. cm^-3, R_sun, R☉).
_UNIT_KEEP_CHARS = r"\w\s/°%\-_^☉"


def normalize_unit_term(t):
    """Canonicalize a MeasurementUnit term using UNIT_ALIASES (grounded in
    this dataset's real vocabulary) rather than generic string cleanup."""
    t = t.strip().lower()
    t = re.sub(rf"[^{_UNIT_KEEP_CHARS}]", "", t)
    t = re.sub(r"\s+", " ", t).strip()
    return UNIT_ALIASES.get(t, t)


def normalize_obs_term(t):
    """Light generic cleanup for ScientificObservable -- this field is open
    vocabulary (869 unique terms in this dataset, 74% singletons), so a
    hand-built alias table isn't feasible; this only strips trivial
    punctuation/plural noise, it does not resolve real synonyms."""
    t = t.strip().lower()
    t = re.sub(r"[^\w\s/°%\-]", "", t)
    t = re.sub(r"\s+", " ", t).strip()
    if len(t) > 3 and t.endswith("s") and not t.endswith("ss"):
        t = t[:-1]
    return t


def set_f1(pred, gold, normalize_fn=None):
    fn = normalize_fn if normalize_fn else (lambda t: t.strip().lower())
    pred_terms = set(fn(t) for t in pred.split(",") if t.strip())
    gold_terms = set(fn(t) for t in gold.split(",") if t.strip())
    if not gold_terms and not pred_terms:
        return 1.0
    if not gold_terms or not pred_terms:
        return 0.0
    overlap = pred_terms & gold_terms
    precision = len(overlap) / len(pred_terms)
    recall = len(overlap) / len(gold_terms)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


# ---------------------------------------------------------------------------
# Data loading (same robust parsing pattern as the other scripts)
# ---------------------------------------------------------------------------

def load_examples(path, log):
    with open(path, encoding="utf-8") as f:
        tasks = json.load(f)

    examples = []
    n_skipped = 0
    for task in tasks:
        task_id = task.get("data", {}).get("id", "unknown")
        text = task.get("data", {}).get("caption_text", "")
        if not text:
            n_skipped += 1
            continue
        annotations = task.get("annotations", [])
        chosen = None
        for ann in annotations:
            if not ann.get("was_cancelled", False):
                chosen = ann
                break
        if chosen is None:
            n_skipped += 1
            continue

        fig_type, unit, obs = None, "", ""
        for r in chosen.get("result", []):
            if r.get("from_name") == "figure_type":
                choices = r["value"].get("choices", [])
                fig_type = choices[0] if choices else None
            elif r.get("from_name") == "measurement_unit":
                unit = r["value"].get("text", [""])[0]
            elif r.get("from_name") == "scientific_observable":
                obs = r["value"].get("text", [""])[0]

        if fig_type not in FIGURE_TYPES:
            n_skipped += 1
            continue

        examples.append({"id": task_id, "text": text, "type": fig_type, "unit": unit, "obs": obs})

    log.append(f"Loaded {len(examples)} usable examples, {n_skipped} skipped (missing/cancelled/invalid).")
    counts = Counter(ex["type"] for ex in examples)
    log.append("Class distribution: " + ", ".join(f"{t}={counts.get(t, 0)}" for t in FIGURE_TYPES))
    return examples


def stratified_split(examples, train_frac=0.8, val_frac=0.1):
    by_label = defaultdict(list)
    for ex in examples:
        by_label[ex["type"]].append(ex)
    train, val, test = [], [], []
    for label, items in by_label.items():
        items = items[:]
        random.shuffle(items)
        n = len(items)
        train_end = int(n * train_frac)
        val_end = int(n * (train_frac + val_frac))
        train.extend(items[:train_end])
        val.extend(items[train_end:val_end])
        test.extend(items[val_end:])
    random.shuffle(train); random.shuffle(val); random.shuffle(test)
    return train, val, test


def stratified_two_way_split(examples, val_frac=0.1):
    """Like stratified_split but only train/val (used inside each k-fold
    iteration to carve a validation set out of that fold's training pool)."""
    by_label = defaultdict(list)
    for ex in examples:
        by_label[ex["type"]].append(ex)
    train, val = [], []
    for label, items in by_label.items():
        items = items[:]
        random.shuffle(items)
        n = len(items)
        val_end = max(1, int(n * val_frac)) if n > 1 else 0
        val.extend(items[:val_end])
        train.extend(items[val_end:])
    random.shuffle(train); random.shuffle(val)
    return train, val


def stratified_kfold(examples, k, seed=SEED):
    """Split into k stratified folds (every example used exactly once as
    test data across all folds -- this is what makes the pooled diagnostic
    plots afterward cover the whole dataset instead of one small split)."""
    by_label = defaultdict(list)
    for ex in examples:
        by_label[ex["type"]].append(ex)
    folds = [[] for _ in range(k)]
    rng = random.Random(seed)
    for label, items in by_label.items():
        items = items[:]
        rng.shuffle(items)
        for i, ex in enumerate(items):
            folds[i % k].append(ex)
    for f in folds:
        rng.shuffle(f)
    return folds


def balance_classes(train_ex, cap_multiplier=2.0):
    """Oversample minority FigureType classes in the TRAIN split only, up to
    min(majority_count, cap_multiplier x own_count). Val/test are never
    touched, so this can only help learning, not inflate reported metrics.
    cap_multiplier bounds how much any one minority class gets duplicated,
    to avoid overfitting on a handful of repeated examples (e.g. scatter=45
    in train would otherwise be duplicated 4x to match time-series=183)."""
    by_label = defaultdict(list)
    for ex in train_ex:
        by_label[ex["type"]].append(ex)
    majority = max(len(v) for v in by_label.values())
    balanced = []
    for label, items in by_label.items():
        target = min(majority, int(cap_multiplier * len(items)))
        reps = items[:]
        while len(reps) < target:
            reps.append(random.choice(items))
        balanced.extend(reps)
    random.shuffle(balanced)
    return balanced


def log_vocabulary_stats(examples, log):
    def terms(s):
        return [t.strip().lower() for t in s.split(",") if t.strip()]
    for field, name in [("unit", "MeasurementUnit"), ("obs", "ScientificObservable")]:
        counter = Counter()
        for ex in examples:
            for t in terms(ex[field]):
                counter[t] += 1
        singles = sum(1 for c in counter.values() if c == 1)
        pct = 100 * singles / len(counter) if counter else 0.0
        log.append(f"{name} vocabulary: {len(counter)} unique terms, {singles} singletons ({pct:.1f}%).")
    return log


def make_compute_metrics(tokenizer):
    """Decodes generations during eval and scores them with the *real*
    metrics (type accuracy + normalized set-F1) instead of eval_loss, so
    checkpoint selection actually tracks what we report at the end."""
    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

        type_correct, unit_f1s, obs_f1s = 0, [], []
        for pred_str, label_str in zip(decoded_preds, decoded_labels):
            p, g = parse_target(pred_str), parse_target(label_str)
            if p["type"] == g["type"]:
                type_correct += 1
            unit_f1s.append(set_f1(p["unit"], g["unit"], normalize_fn=normalize_unit_term))
            obs_f1s.append(set_f1(p["obs"], g["obs"], normalize_fn=normalize_obs_term))

        n = max(len(decoded_preds), 1)
        type_acc = type_correct / n
        unit_f1 = float(np.mean(unit_f1s)) if unit_f1s else 0.0
        obs_f1 = float(np.mean(obs_f1s)) if obs_f1s else 0.0
        return {
            "type_acc": type_acc,
            "unit_f1_norm": unit_f1,
            "obs_f1_norm": obs_f1,
            "combined_score": (type_acc + unit_f1 + obs_f1) / 3,
        }
    return compute_metrics


def build_hf_dataset(tokenizer, exs):
    rows = []
    for ex in exs:
        input_text = PROMPT_PREFIX + ex["text"]
        target_text = build_target(ex["type"], ex["unit"], ex["obs"])
        # No padding here -- the data collator pads per-batch (dynamic
        # padding), which avoids wasting compute on short captions.
        input_enc = tokenizer(input_text, truncation=True, max_length=MAX_INPUT_LENGTH)
        target_enc = tokenizer(target_text, truncation=True, max_length=MAX_TARGET_LENGTH)
        rows.append({
            "input_ids": input_enc["input_ids"],
            "attention_mask": input_enc["attention_mask"],
            "labels": target_enc["input_ids"],
        })
    return Dataset.from_list(rows)


def write_inference_config(model_dir, model_name, epochs, lr, model_size):
    """Writes everything needed to actually *use* the saved model later,
    without needing this training script around: the exact prompt prefix
    (mismatched prefixes silently degrade generation), the target string
    format + how to parse it back into fields, the unit alias table, the
    plural/punctuation cleanup rule for observables, and the generation
    length cap used during training/eval."""
    cfg = {
        "base_model": model_name,
        "model_size": model_size,
        "prompt_prefix": PROMPT_PREFIX,
        "max_input_length": MAX_INPUT_LENGTH,
        "max_target_length": MAX_TARGET_LENGTH,
        "target_format": "type: <FigureType> | units: <comma-separated units, or 'none'> | observable: <comma-separated observables, or 'none'>",
        "figure_types": FIGURE_TYPES,
        "unit_aliases": UNIT_ALIASES,
        "unit_normalization_notes": "lowercase, strip chars outside [\\w\\s/°%\\-_^☉], collapse whitespace, then map through unit_aliases",
        "observable_normalization_notes": "lowercase, strip chars outside [\\w\\s/°%\\-], collapse whitespace, strip trailing plural 's' (len>3, not already ending 'ss')",
        "training_epochs_requested": epochs,
        "learning_rate": lr,
        "note": "Reload with AutoTokenizer/AutoModelForSeq2SeqLM.from_pretrained(this_dir). "
                "Prepend prompt_prefix to captions before tokenizing. Parse generations with "
                "the target_format above (split on '|', then ':'; 'none' means empty list).",
    }
    with open(Path(model_dir) / "inference_config.json", "w") as f:
        json.dump(cfg, f, indent=2)
    print(f"  saved: inference_config.json (in {model_dir})")


def train_and_predict(model_name, train_ex, val_ex, test_ex, batch_size, grad_accum,
                       grad_checkpointing, epochs, lr, use_bf16, use_fp16,
                       checkpoint_dir, save_final_model_to=None, model_size=None):
    """One full train+predict cycle. Used both for the default single-split
    run and for each fold in --kfold mode (a fresh model is always loaded
    from the pretrained checkpoint here -- folds never share learned weights)."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    if grad_checkpointing:
        model.gradient_checkpointing_enable()
        model.config.use_cache = False  # required alongside grad checkpointing

    dataset = DatasetDict({
        "train": build_hf_dataset(tokenizer, train_ex),
        "validation": build_hf_dataset(tokenizer, val_ex),
        "test": build_hf_dataset(tokenizer, test_ex),
    })

    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True, label_pad_token_id=-100)

    training_args = Seq2SeqTrainingArguments(
        output_dir=str(checkpoint_dir),
        eval_strategy="epoch", save_strategy="epoch",
        learning_rate=lr,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        num_train_epochs=epochs, weight_decay=0.01, load_best_model_at_end=True,
        metric_for_best_model="combined_score", greater_is_better=True,
        logging_steps=10, report_to="none", predict_with_generate=True,
        generation_max_length=MAX_TARGET_LENGTH,
        generation_num_beams=4,
        bf16=use_bf16, fp16=use_fp16,
        save_total_limit=1, save_only_model=True, seed=SEED,
    )

    trainer = Seq2SeqTrainer(
        model=model, args=training_args, train_dataset=dataset["train"], eval_dataset=dataset["validation"],
        processing_class=tokenizer, data_collator=data_collator,
        compute_metrics=make_compute_metrics(tokenizer),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
    )

    t0 = time.time()
    trainer.train()
    elapsed_min = (time.time() - t0) / 60

    predictions = trainer.predict(
        dataset["test"], 
        max_length=MAX_TARGET_LENGTH,
        num_beams=4
    )
    pred_ids = np.where(predictions.predictions != -100, predictions.predictions, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)

    if save_final_model_to:
        trainer.save_model(str(save_final_model_to))
        tokenizer.save_pretrained(str(save_final_model_to))
        write_inference_config(save_final_model_to, model_name, epochs, lr, model_size)

    return trainer, decoded_preds, elapsed_min


def evaluate_predictions(test_ex, decoded_preds):
    """Turns raw decoded generations + gold examples into every metric this
    script reports. Shared by the single-split path and the k-fold path so
    both are scored identically."""
    gold_types = [ex["type"] for ex in test_ex]
    gold_units = [ex["unit"] for ex in test_ex]
    gold_obs = [ex["obs"] for ex in test_ex]

    pred_types, pred_units, pred_obs = [], [], []
    for p in decoded_preds:
        parsed = parse_target(p)
        pred_types.append(parsed["type"])
        pred_units.append(parsed["unit"])
        pred_obs.append(parsed["obs"])

    valid_type_mask = [pt in FIGURE_TYPES for pt in pred_types]
    n_invalid_type = len(valid_type_mask) - sum(valid_type_mask)
    clean_gold_types = [gold_types[i] for i in range(len(gold_types)) if valid_type_mask[i]]
    clean_pred_types = [pred_types[i] for i in range(len(pred_types)) if valid_type_mask[i]]

    unit_f1s_raw = [set_f1(pu, gu) for pu, gu in zip(pred_units, gold_units)]
    obs_f1s_raw = [set_f1(po, go) for po, go in zip(pred_obs, gold_obs)]
    unit_f1s_norm = [set_f1(pu, gu, normalize_fn=normalize_unit_term) for pu, gu in zip(pred_units, gold_units)]
    obs_f1s_norm = [set_f1(po, go, normalize_fn=normalize_obs_term) for po, go in zip(pred_obs, gold_obs)]

    type_acc = (sum(1 for g, p in zip(clean_gold_types, clean_pred_types) if g == p)
                / max(len(clean_gold_types), 1))

    return {
        "gold_types": gold_types, "gold_units": gold_units, "gold_obs": gold_obs,
        "pred_types": pred_types, "pred_units": pred_units, "pred_obs": pred_obs,
        "valid_type_mask": valid_type_mask, "n_invalid_type": n_invalid_type,
        "clean_gold_types": clean_gold_types, "clean_pred_types": clean_pred_types,
        "unit_f1s_raw": unit_f1s_raw, "obs_f1s_raw": obs_f1s_raw,
        "unit_f1s_norm": unit_f1s_norm, "obs_f1s_norm": obs_f1s_norm,
        "type_acc": type_acc,
    }


def make_diagnostic_plots(plots_dir, res, train_ex, val_ex, test_ex, trainer=None, class_dist_by_split=True):
    """All diagnostic plots, shared by the single-split run and the k-fold
    pooled-out-of-fold run. trainer is only available (and only plotted) in
    single-split mode, since k-fold trains several separate models."""
    clean_gold_types, clean_pred_types = res["clean_gold_types"], res["clean_pred_types"]
    unit_f1s_raw, obs_f1s_raw = res["unit_f1s_raw"], res["obs_f1s_raw"]
    unit_f1s_norm, obs_f1s_norm = res["unit_f1s_norm"], res["obs_f1s_norm"]
    width = 0.25

    # 1. Class distribution
    fig, ax = plt.subplots(figsize=(9, 5.5))
    x = np.arange(len(FIGURE_TYPES))
    if class_dist_by_split:
        for i, (name, exs) in enumerate([("train", train_ex), ("val", val_ex), ("test", test_ex)]):
            counts = Counter(e["type"] for e in exs)
            vals = [counts.get(t, 0) for t in FIGURE_TYPES]
            ax.bar(x + (i - 1) * width, vals, width, label=name, color=PALETTE[i])
        ax.legend(**LEGEND_OUTSIDE)
    else:
        counts = Counter(e["type"] for e in test_ex)
        vals = [counts.get(t, 0) for t in FIGURE_TYPES]
        ax.bar(x, vals, 0.5, color=PALETTE[0])
    ax.set_xticks(x); ax.set_xticklabels(FIGURE_TYPES, rotation=20, ha="right")
    ax.set_ylabel("Count"); ax.set_title("FigureType class distribution")
    savefig(fig, plots_dir / "class_distribution.png")

    # 2. Confusion matrix
    cm = confusion_matrix(clean_gold_types, clean_pred_types, labels=FIGURE_TYPES)
    fig, ax = plt.subplots(figsize=(8, 6.5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=FIGURE_TYPES, yticklabels=FIGURE_TYPES,
                ax=ax, linewidths=1, linecolor="white", cbar_kws={"label": "Count"})
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title("Generative model — FigureType confusion matrix")
    plt.xticks(rotation=25, ha="right")
    savefig(fig, plots_dir / "generative_figuretype_confusion_matrix.png")

    # 3. Per-class precision/recall/F1
    per_class_p, per_class_r, per_class_f1, _ = precision_recall_fscore_support(
        clean_gold_types, clean_pred_types, labels=FIGURE_TYPES, zero_division=0
    )
    fig, ax = plt.subplots(figsize=(9, 5.5))
    x = np.arange(len(FIGURE_TYPES))
    ax.bar(x - width, per_class_p, width, label="Precision", color=PALETTE[0])
    ax.bar(x, per_class_r, width, label="Recall", color=PALETTE[1])
    ax.bar(x + width, per_class_f1, width, label="F1", color=PALETTE[2])
    ax.set_xticks(x); ax.set_xticklabels(FIGURE_TYPES, rotation=20, ha="right")
    ax.set_ylim(0, 1.08); ax.set_ylabel("Score")
    ax.set_title("FigureType per-class precision / recall / F1")
    ax.legend(**LEGEND_OUTSIDE)
    savefig(fig, plots_dir / "generative_figuretype_per_class_metrics.png")

    # 4. Set-F1 raw vs normalized
    fig, ax = plt.subplots(figsize=(8, 5.5))
    labels_ = ["MeasurementUnit", "ScientificObservable"]
    raw_vals = [np.mean(unit_f1s_raw), np.mean(obs_f1s_raw)]
    norm_vals = [np.mean(unit_f1s_norm), np.mean(obs_f1s_norm)]
    x = np.arange(len(labels_))
    ax.bar(x - width / 2, raw_vals, width, label="Raw set-F1", color=PALETTE[3])
    ax.bar(x + width / 2, norm_vals, width, label="Normalized set-F1", color=PALETTE[0])
    for xi, v in zip(x - width / 2, raw_vals):
        ax.text(xi, v + 0.02, f"{v:.3f}", ha="center", fontsize=10)
    for xi, v in zip(x + width / 2, norm_vals):
        ax.text(xi, v + 0.02, f"{v:.3f}", ha="center", fontsize=10)
    ax.set_xticks(x); ax.set_xticklabels(labels_)
    ax.set_ylim(0, 1.15); ax.set_ylabel("Mean set-overlap F1")
    ax.set_title("Generative field quality: raw vs. normalized set-F1")
    ax.legend(**LEGEND_OUTSIDE)
    savefig(fig, plots_dir / "generative_set_f1_raw_vs_normalized.png")

    # 5. Vocabulary richness
    def term_counts(field):
        c = Counter()
        for ex in train_ex + val_ex + test_ex:
            for t in ex[field].split(","):
                t = t.strip().lower()
                if t:
                    c[t] += 1
        return c
    unit_vocab = term_counts("unit")
    obs_vocab = term_counts("obs")
    fig, ax = plt.subplots(figsize=(8, 5.5))
    fields_ = ["MeasurementUnit", "ScientificObservable"]
    uniq = [len(unit_vocab), len(obs_vocab)]
    singles = [sum(1 for c in unit_vocab.values() if c == 1), sum(1 for c in obs_vocab.values() if c == 1)]
    x = np.arange(len(fields_))
    ax.bar(x - width / 2, uniq, width, label="Unique terms", color=PALETTE[0])
    ax.bar(x + width / 2, singles, width, label="Singleton terms (seen once)", color=PALETTE[2])
    for xi, v in zip(x - width / 2, uniq):
        ax.text(xi, v + max(uniq) * 0.02, str(v), ha="center", fontsize=10)
    for xi, v in zip(x + width / 2, singles):
        ax.text(xi, v + max(uniq) * 0.02, str(v), ha="center", fontsize=10)
    ax.set_xticks(x); ax.set_xticklabels(fields_)
    ax.set_ylabel("Count"); ax.set_title("Vocabulary richness — why observable F1 is harder")
    ax.legend(**LEGEND_OUTSIDE)
    savefig(fig, plots_dir / "vocabulary_richness.png")

    # 6. Distribution of normalized set-F1
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(unit_f1s_norm, bins=15, alpha=0.6, label="MeasurementUnit", color=PALETTE[0])
    ax.hist(obs_f1s_norm, bins=15, alpha=0.6, label="ScientificObservable", color=PALETTE[1])
    ax.set_xlabel("Normalized set-overlap F1 (per example)"); ax.set_ylabel("Count")
    ax.set_title("Distribution of generation quality")
    ax.legend(**LEGEND_OUTSIDE)
    savefig(fig, plots_dir / "generative_set_f1_distribution.png")

    # 7. Training curves (single-split mode only -- k-fold trains several models)
    if trainer is not None:
        history = trainer.state.log_history
        train_steps, train_loss = [], []
        eval_epochs, eval_loss, eval_combined = [], [], []
        for entry in history:
            if "loss" in entry and "epoch" in entry and "eval_loss" not in entry:
                train_steps.append(entry["epoch"]); train_loss.append(entry["loss"])
            if "eval_loss" in entry:
                eval_epochs.append(entry["epoch"]); eval_loss.append(entry["eval_loss"])
                eval_combined.append(entry.get("eval_combined_score", np.nan))

        fig, ax1 = plt.subplots(figsize=(9, 5.5))
        ax1.plot(train_steps, train_loss, color=PALETTE[0], label="Train loss", alpha=0.8)
        ax1.plot(eval_epochs, eval_loss, color=PALETTE[2], marker="o", label="Eval loss")
        ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
        ax2 = ax1.twinx()
        ax2.plot(eval_epochs, eval_combined, color=PALETTE[1], marker="s", linestyle="--", label="Eval combined score")
        ax2.set_ylabel("Combined score (type acc + unit F1 + obs F1) / 3")
        ax2.set_ylim(0, 1.05)
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", bbox_to_anchor=(1.08, 1), frameon=False)
        ax1.set_title("Training curves")
        savefig(fig, plots_dir / "generative_training_curves.png")


def write_metrics_summary(output_folder, res, args, elapsed_min, n_test, extra=None):
    summary = {
        "model_size": args.model_size,
        "training_minutes": round(elapsed_min, 1),
        "figuretype_accuracy": res["type_acc"],
        "measurementunit_mean_set_f1_raw": float(np.mean(res["unit_f1s_raw"])),
        "measurementunit_mean_set_f1_normalized": float(np.mean(res["unit_f1s_norm"])),
        "scientificobservable_mean_set_f1_raw": float(np.mean(res["obs_f1s_raw"])),
        "scientificobservable_mean_set_f1_normalized": float(np.mean(res["obs_f1s_norm"])),
        "n_invalid_figuretype_generations": res["n_invalid_type"],
        "n_test_examples": n_test,
    }
    if extra:
        summary.update(extra)
    print("\nSummary:", json.dumps(summary, indent=2))
    with open(output_folder / "generative_metrics.json", "w") as f:
        json.dump(summary, f, indent=2)
    return summary


def write_example_predictions(output_folder, test_ex, decoded_preds, res, n=15):
    examples_out = []
    for i in range(min(n, len(test_ex))):
        examples_out.append({
            "caption": test_ex[i]["text"][:150],
            "gold": build_target(res["gold_types"][i], res["gold_units"][i], res["gold_obs"][i]),
            "predicted": decoded_preds[i],
        })
    with open(output_folder / "generative_example_predictions.json", "w") as f:
        json.dump(examples_out, f, indent=2)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("output_folder")
    parser.add_argument("--data_file", default="figures_export.json",
                         help="Filename of the annotation export, looked up inside output_folder "
                              "(e.g. Heliophysics_Figure_Caption_Annotation.json).")
    parser.add_argument("--model_size", choices=["base", "large"], default="base",
                         help="FLAN-T5 size. 'base' (~250M) is fast and safe on any free GPU. "
                              "'large' (~780M) usually gives better generation quality but needs "
                              "the smaller batch size / grad accumulation defaults below.")
    parser.add_argument("--epochs", type=int, default=15)
    parser.add_argument("--batch_size", type=int, default=None, help="Default depends on --model_size.")
    parser.add_argument("--grad_accum", type=int, default=None, help="Default depends on --model_size.")
    parser.add_argument("--lr", type=float, default=3e-4)
    parser.add_argument("--balance_classes", action=argparse.BooleanOptionalAction, default=True,
                         help="Oversample minority FigureType classes in the train split only "
                              "(this dataset is ~4x imbalanced: time-series=229 vs scatter=57). "
                              "Use --no-balance_classes to disable.")
    parser.add_argument("--kfold", type=int, default=1,
                         help="If >1, run k-fold cross-validation instead of a single split: "
                              "trains k separate models (each on a different 1/k held-out fold) "
                              "to report mean+-std metrics and pooled out-of-fold diagnostics "
                              "covering the WHOLE dataset, instead of one small test split. "
                              "This multiplies total runtime by roughly k -- e.g. --kfold 5 on "
                              "the base model takes ~5x a single run (still fits Kaggle's free "
                              "weekly GPU quota, but budget the time). Default 1 = off (single split).")
    args = parser.parse_args()

    defaults = DEFAULTS[args.model_size]
    batch_size = args.batch_size or defaults["batch_size"]
    grad_accum = args.grad_accum or defaults["grad_accum"]
    grad_checkpointing = defaults["grad_checkpointing"]
    model_name = MODEL_NAMES[args.model_size]

    output_folder = Path(args.output_folder)
    figures_path = output_folder / args.data_file
    plots_dir = output_folder / "generative_plots"
    plots_dir.mkdir(parents=True, exist_ok=True)

    if not figures_path.exists():
        print(f"Not found: {figures_path}")
        sys.exit(1)

    has_gpu = torch.cuda.is_available()
    use_bf16 = has_gpu and torch.cuda.is_bf16_supported()
    use_fp16 = has_gpu and not use_bf16
    if not has_gpu:
        print("WARNING: no GPU detected. This will be slow on CPU -- use a GPU runtime "
              "(Colab: Runtime > Change runtime type; Kaggle: Settings > Accelerator).")
    else:
        gpu_name = torch.cuda.get_device_name(0)
        print(f"GPU: {gpu_name} | precision: {'bf16' if use_bf16 else 'fp16'}")

    print(f"Model: {model_name} | effective batch size: {batch_size * grad_accum} "
          f"(per-device={batch_size} x grad_accum={grad_accum}) | grad checkpointing: {grad_checkpointing}")

    log = []
    examples = load_examples(figures_path, log)
    log_vocabulary_stats(examples, log)
    for line in log:
        print(line)
    with open(output_folder / "generative_prep_report.txt", "w") as f:
        f.write("\n".join(log))

    # =========================================================================
    # Mode 1 (default): single stratified train/val/test split
    # =========================================================================
    if args.kfold <= 1:
        train_ex, val_ex, test_ex = stratified_split(examples)
        print(f"Stratified split: train={len(train_ex)} val={len(val_ex)} test={len(test_ex)}")

        if args.balance_classes:
            pre_counts = Counter(e["type"] for e in train_ex)
            train_ex = balance_classes(train_ex)
            post_counts = Counter(e["type"] for e in train_ex)
            print(f"Class balancing ON: train {sum(pre_counts.values())} -> {sum(post_counts.values())} "
                  f"examples. Per-class: {dict(pre_counts)} -> {dict(post_counts)}")
        else:
            print("Class balancing OFF (--no-balance_classes).")

        print("\nTraining...")
        trainer, decoded_preds, elapsed_min = train_and_predict(
            model_name, train_ex, val_ex, test_ex, batch_size, grad_accum, grad_checkpointing,
            args.epochs, args.lr, use_bf16, use_fp16,
            checkpoint_dir=output_folder / "generative_model_checkpoints",
            save_final_model_to=output_folder / "generative_model_final",
            model_size=args.model_size,
        )
        print(f"\nTraining finished in {elapsed_min:.1f} minutes.")

        res = evaluate_predictions(test_ex, decoded_preds)
        print(f"\nGenerated FigureType values outside the valid 5 classes: "
              f"{res['n_invalid_type']}/{len(decoded_preds)}")

        type_report = classification_report(res["clean_gold_types"], res["clean_pred_types"], labels=FIGURE_TYPES,
                                            target_names=FIGURE_TYPES, digits=3, zero_division=0)
        print(type_report)
        with open(output_folder / "generative_figuretype_report.txt", "w") as f:
            f.write(type_report)

        write_metrics_summary(output_folder, res, args, elapsed_min, len(test_ex))
        write_example_predictions(output_folder, test_ex, decoded_preds, res)
        make_diagnostic_plots(plots_dir, res, train_ex, val_ex, test_ex, trainer=trainer, class_dist_by_split=True)

        print(f"\nDone. Model saved to: {output_folder / 'generative_model_final'}")
        print(f"All plots saved to: {plots_dir}")
        return

    # =========================================================================
    # Mode 2 (optional, --kfold N): k-fold cross-validation
    #
    # Trains N separate models, each held out on a different 1/N slice of the
    # data. This does NOT make any single model better -- it tells you how
    # much your reported numbers would move if you'd happened to draw a
    # different test split, which matters here because 672 examples is small
    # enough that a single split's numbers carry real sampling noise. As a
    # side benefit, the pooled out-of-fold predictions cover all 672
    # examples (each scored by a model that never trained on it), so the
    # diagnostic plots below are less noisy than the single-split versions.
    # No final "generative_model_final" is saved in this mode -- rerun
    # without --kfold to produce the actual deliverable model.
    # =========================================================================
    print(f"\nRunning {args.kfold}-fold cross-validation. This trains {args.kfold} separate "
          f"models and will take roughly {args.kfold}x as long as a single run.")
    folds = stratified_kfold(examples, k=args.kfold)

    per_fold_metrics = []
    oof_test_ex, oof_decoded_preds = [], []

    for i in range(args.kfold):
        test_ex_i = folds[i]
        train_pool = [ex for j, f in enumerate(folds) if j != i for ex in f]
        train_ex_i, val_ex_i = stratified_two_way_split(train_pool, val_frac=0.1)

        if args.balance_classes:
            train_ex_i = balance_classes(train_ex_i)

        print(f"\n=== Fold {i + 1}/{args.kfold}: train={len(train_ex_i)} "
              f"val={len(val_ex_i)} test={len(test_ex_i)} ===")

        fold_ckpt_dir = output_folder / f"kfold_checkpoints_tmp_{i}"
        _, decoded_preds_i, elapsed_i = train_and_predict(
            model_name, train_ex_i, val_ex_i, test_ex_i, batch_size, grad_accum, grad_checkpointing,
            args.epochs, args.lr, use_bf16, use_fp16,
            checkpoint_dir=fold_ckpt_dir, save_final_model_to=None,
        )
        shutil.rmtree(fold_ckpt_dir, ignore_errors=True)  # don't accumulate disk across folds

        res_i = evaluate_predictions(test_ex_i, decoded_preds_i)
        fold_summary = {
            "fold": i + 1,
            "n_test": len(test_ex_i),
            "training_minutes": round(elapsed_i, 1),
            "figuretype_accuracy": res_i["type_acc"],
            "measurementunit_f1_normalized": float(np.mean(res_i["unit_f1s_norm"])),
            "scientificobservable_f1_normalized": float(np.mean(res_i["obs_f1s_norm"])),
        }
        per_fold_metrics.append(fold_summary)
        print(json.dumps(fold_summary, indent=2))

        oof_test_ex.extend(test_ex_i)
        oof_decoded_preds.extend(decoded_preds_i)

    def agg(key):
        vals = [f[key] for f in per_fold_metrics]
        return float(np.mean(vals)), float(np.std(vals))

    kfold_summary = {"k": args.kfold, "per_fold": per_fold_metrics}
    for key in ["figuretype_accuracy", "measurementunit_f1_normalized", "scientificobservable_f1_normalized"]:
        m, s = agg(key)
        kfold_summary[f"{key}_mean"] = m
        kfold_summary[f"{key}_std"] = s
    print("\nK-fold summary:", json.dumps(kfold_summary, indent=2))
    with open(output_folder / "kfold_summary.json", "w") as f:
        json.dump(kfold_summary, f, indent=2)

    # Variance plot: mean +- std across folds for the three headline metrics
    fig, ax = plt.subplots(figsize=(9, 5.5))
    metric_keys = ["figuretype_accuracy", "measurementunit_f1_normalized", "scientificobservable_f1_normalized"]
    metric_labels = ["FigureType Acc", "MeasurementUnit F1", "ScientificObservable F1"]
    means = [kfold_summary[f"{k}_mean"] for k in metric_keys]
    stds = [kfold_summary[f"{k}_std"] for k in metric_keys]
    x = np.arange(len(metric_keys))
    ax.bar(x, means, yerr=stds, capsize=6, color=PALETTE[:3])
    for xi, mval, sval in zip(x, means, stds):
        ax.text(xi, mval + sval + 0.02, f"{mval:.3f} ± {sval:.3f}", ha="center", fontsize=10)
    ax.set_xticks(x); ax.set_xticklabels(metric_labels, rotation=10)
    ax.set_ylim(0, 1.15); ax.set_ylabel("Score")
    ax.set_title(f"{args.kfold}-fold cross-validation: mean ± std across folds")
    savefig(fig, plots_dir / "kfold_variance.png")

    # Pooled out-of-fold diagnostics -- every example scored exactly once by
    # a model that never trained on it, covering the whole dataset.
    res = evaluate_predictions(oof_test_ex, oof_decoded_preds)
    type_report = classification_report(res["clean_gold_types"], res["clean_pred_types"], labels=FIGURE_TYPES,
                                        target_names=FIGURE_TYPES, digits=3, zero_division=0)
    print("\nPooled out-of-fold FigureType report (covers all examples):")
    print(type_report)
    with open(output_folder / "generative_figuretype_report.txt", "w") as f:
        f.write(type_report)

    total_minutes = sum(f["training_minutes"] for f in per_fold_metrics)
    write_metrics_summary(output_folder, res, args, total_minutes, len(oof_test_ex),
                           extra={"kfold": args.kfold, "note": "pooled out-of-fold metrics across all folds"})
    write_example_predictions(output_folder, oof_test_ex, oof_decoded_preds, res)
    make_diagnostic_plots(plots_dir, res, oof_test_ex, [], oof_test_ex, trainer=None, class_dist_by_split=False)

    print(f"\nDone. No single final model is saved in --kfold mode -- rerun without --kfold "
          f"to train and save the actual deliverable model.")
    print(f"All plots saved to: {plots_dir}")


if __name__ == "__main__":
    main()

Writing train_generative_figure_model.py


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Paths (edit to match your environment)
DATA_FILE = "data/Heliophysics_Figure_Caption_Annotated.json"  # annotated captions (not included in this repo)
OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)
os.environ["DATA_FILE"] = DATA_FILE
os.environ["OUT_DIR"] = OUT_DIR


In [3]:
!python "train_generative_figure_model.py" "$OUT_DIR" --data_file "$DATA_FILE" --epochs 30 --model_size large

GPU: Tesla T4 | precision: bf16
Model: google/flan-t5-large | effective batch size: 16 (per-device=2 x grad_accum=8) | grad checkpointing: True
Loaded 672 usable examples, 0 skipped (missing/cancelled/invalid).
Class distribution: time-series=229, spectral=76, spatial-map=209, scatter=57, other=101
MeasurementUnit vocabulary: 83 unique terms, 33 singletons (39.8%).
ScientificObservable vocabulary: 868 unique terms, 643 singletons (74.1%).
Stratified split: train=535 val=68 test=69
Class balancing ON: train 535 -> 736 examples. Per-class: {'other': 80, 'spatial-map': 167, 'spectral': 60, 'time-series': 183, 'scatter': 45} -> {'spatial-map': 183, 'time-series': 183, 'other': 160, 'spectral': 120, 'scatter': 90}
Training...
[per-epoch validation lines condensed]
{'eval_loss': '0.8706', 'eval_type_acc': '0.7941', 'eval_unit_f1_norm': '0.7712', 'eval_obs_f1_norm': '0.4384', 'eval_combined_score': '0.6679', 'eval_runtime': '85.03', 'eval_samples_per_second': '0.8', 'eval_steps_per_second': '

In [4]:
%%writefile seen_vs_unseen_breakdown.py

"""
Seen-vs-unseen vocabulary breakdown for MeasurementUnit / ScientificObservable.

Does NOT retrain anything -- loads your already-trained model
(generative_model_final/) and runs it on the test set, then buckets each
test example by whether its gold terms were present in the TRAINING
vocabulary. This proves (or disproves) whether low ScientificObservable
scores are a data-coverage ceiling rather than a modeling failure.

Reproduces the EXACT same train/val/test split as the original training
script (same seed, same data file, same splitting logic) so the "test set"
here is identical to the one the model was actually evaluated on.

Usage:
    python seen_vs_unseen_breakdown.py /path/to/output_folder --data_file <file>
"""

import sys
import json
import random
import argparse
import re
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

FIGURE_TYPES = ["time-series", "spectral", "spatial-map", "scatter", "other"]
SEED = 42
MAX_INPUT_LENGTH = 300
MAX_TARGET_LENGTH = 100
PROMPT_PREFIX = "Extract figure attributes from this caption: "

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

sns.set_theme(style="white", context="talk", font_scale=0.72)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
PALETTE = ["#0072B2", "#009E73", "#D55E00", "#CC79A7", "#E69F00", "#56B4E9"]
sns.set_palette(sns.color_palette(PALETTE))
LEGEND_OUTSIDE = dict(loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False)


# ---------------------------------------------------------------------------
# Exact copies of the original script's logic, so the split/parsing/metric
# are guaranteed identical -- not re-derived, not approximated.
# ---------------------------------------------------------------------------

def build_target(fig_type, unit, obs):
    unit = unit if unit else "none"
    obs = obs if obs else "none"
    return f"type: {fig_type} | units: {unit} | observable: {obs}"


def parse_target(s):
    parts = {}
    for chunk in s.split("|"):
        if ":" in chunk:
            k, v = chunk.split(":", 1)
            parts[k.strip()] = v.strip()
    unit = parts.get("units", "")
    obs = parts.get("observable", "")
    return {"type": parts.get("type", ""),
            "unit": "" if unit == "none" else unit,
            "obs": "" if obs == "none" else obs}


UNIT_ALIASES = {
    "au": "au", "degree": "degree", "degrees": "degree", "deg": "degree", "°": "degree",
    "minutes": "min", "minute": "min", "min": "min", "hr": "hr", "hour": "hr", "hours": "hr",
    "r_sun": "r_sun", "rs": "r_sun", "r☉": "r_sun", "solar radii": "r_sun", "solar radius": "r_sun",
    "r_e": "r_e", "re": "r_e", "earth radii": "r_e", "earth radius": "r_e",
    "ut": "ut", "k": "k", "mk": "mk", "kev": "kev", "ev": "ev", "mev": "mev",
}
_UNIT_KEEP_CHARS = r"\w\s/°%\-_^☉"


def normalize_unit_term(t):
    t = t.strip().lower()
    t = re.sub(rf"[^{_UNIT_KEEP_CHARS}]", "", t)
    t = re.sub(r"\s+", " ", t).strip()
    return UNIT_ALIASES.get(t, t)


def normalize_obs_term(t):
    t = t.strip().lower()
    t = re.sub(r"[^\w\s/°%\-]", "", t)
    t = re.sub(r"\s+", " ", t).strip()
    if len(t) > 3 and t.endswith("s") and not t.endswith("ss"):
        t = t[:-1]
    return t


def set_f1(pred, gold, normalize_fn):
    pred_terms = set(normalize_fn(t) for t in pred.split(",") if t.strip())
    gold_terms = set(normalize_fn(t) for t in gold.split(",") if t.strip())
    if not gold_terms and not pred_terms:
        return 1.0
    if not gold_terms or not pred_terms:
        return 0.0
    overlap = pred_terms & gold_terms
    precision = len(overlap) / len(pred_terms)
    recall = len(overlap) / len(gold_terms)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def word_level_f1(pred, gold):
    """Lenient partial-credit metric: splits BOTH the full pred and gold
    strings into individual words (ignoring comma structure), then computes
    F1 over word overlap. 'solar wind speed' vs 'solar wind velocity' scores
    2/3 overlap here instead of 0 under strict phrase matching -- this
    isolates whether low strict-F1 is really about wrong answers, or just
    paraphrasing of otherwise-correct answers."""
    def words(s):
        return set(w for w in re.sub(r"[^\w\s]", " ", s.lower()).split() if len(w) > 2)
    pred_w, gold_w = words(pred), words(gold)
    if not gold_w and not pred_w:
        return 1.0
    if not gold_w or not pred_w:
        return 0.0
    overlap = pred_w & gold_w
    precision = len(overlap) / len(pred_w)
    recall = len(overlap) / len(gold_w)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def load_examples(path):
    with open(path, encoding="utf-8") as f:
        tasks = json.load(f)
    examples = []
    for task in tasks:
        task_id = task.get("data", {}).get("id", "unknown")
        text = task.get("data", {}).get("caption_text", "")
        if not text:
            continue
        annotations = task.get("annotations", [])
        chosen = None
        for ann in annotations:
            if not ann.get("was_cancelled", False):
                chosen = ann
                break
        if chosen is None:
            continue
        fig_type, unit, obs = None, "", ""
        for r in chosen.get("result", []):
            if r.get("from_name") == "figure_type":
                choices = r["value"].get("choices", [])
                fig_type = choices[0] if choices else None
            elif r.get("from_name") == "measurement_unit":
                unit = r["value"].get("text", [""])[0]
            elif r.get("from_name") == "scientific_observable":
                obs = r["value"].get("text", [""])[0]
        if fig_type not in FIGURE_TYPES:
            continue
        examples.append({"id": task_id, "text": text, "type": fig_type, "unit": unit, "obs": obs})
    return examples


def stratified_split(examples, train_frac=0.8, val_frac=0.1):
    by_label = defaultdict(list)
    for ex in examples:
        by_label[ex["type"]].append(ex)
    train, val, test = [], [], []
    for label, items in by_label.items():
        items = items[:]
        random.shuffle(items)
        n = len(items)
        train_end = int(n * train_frac)
        val_end = int(n * (train_frac + val_frac))
        train.extend(items[:train_end])
        val.extend(items[train_end:val_end])
        test.extend(items[val_end:])
    random.shuffle(train); random.shuffle(val); random.shuffle(test)
    return train, val, test


def savefig(fig, path):
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved: {path.name}")


def bucket_examples(test_ex, train_vocab, field, normalize_fn):
    """Returns {example_index: 'fully_seen' | 'partially_seen' | 'fully_unseen'}
    based on whether each test example's gold terms for `field` appear in
    train_vocab. Examples with empty gold value are excluded (no bucket)."""
    buckets = {}
    for i, ex in enumerate(test_ex):
        gold_terms = set(normalize_fn(t) for t in ex[field].split(",") if t.strip())
        if not gold_terms:
            continue
        seen = gold_terms & train_vocab
        if len(seen) == len(gold_terms):
            buckets[i] = "fully_seen"
        elif len(seen) > 0:
            buckets[i] = "partially_seen"
        else:
            buckets[i] = "fully_unseen"
    return buckets


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("output_folder")
    parser.add_argument("--data_file", default="figures_export.json")
    args = parser.parse_args()

    output_folder = Path(args.output_folder)
    figures_path = output_folder / args.data_file
    model_path = output_folder / "generative_model_final"
    plots_dir = output_folder / "generative_plots"
    plots_dir.mkdir(exist_ok=True)

    if not figures_path.exists():
        print(f"Not found: {figures_path}"); sys.exit(1)
    if not model_path.exists():
        print(f"Model not found at {model_path} -- train it first (train_generative_figure_model.py).")
        sys.exit(1)

    print("Reproducing the exact train/val/test split used during training...")
    examples = load_examples(figures_path)
    train_ex, val_ex, test_ex = stratified_split(examples)
    print(f"train={len(train_ex)} val={len(val_ex)} test={len(test_ex)}")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading model from {model_path} onto {device}...")
    tokenizer = AutoTokenizer.from_pretrained(str(model_path))
    model = AutoModelForSeq2SeqLM.from_pretrained(str(model_path)).to(device).eval()

    print(f"Generating predictions for {len(test_ex)} test captions...")
    decoded_preds = []
    batch_size = 8
    for i in range(0, len(test_ex), batch_size):
        batch = test_ex[i:i + batch_size]
        inputs = tokenizer([PROMPT_PREFIX + ex["text"] for ex in batch],
                            return_tensors="pt", truncation=True, max_length=MAX_INPUT_LENGTH,
                            padding=True).to(device)
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_length=MAX_TARGET_LENGTH)
        decoded_preds.extend(tokenizer.batch_decode(output_ids, skip_special_tokens=True))
        print(f"  {min(i + batch_size, len(test_ex))}/{len(test_ex)}")

    parsed_preds = [parse_target(p) for p in decoded_preds]

    # --- build training vocabularies (per field, normalized) ----------------
    unit_train_vocab = set()
    obs_train_vocab = set()
    for ex in train_ex:
        for t in ex["unit"].split(","):
            if t.strip():
                unit_train_vocab.add(normalize_unit_term(t))
        for t in ex["obs"].split(","):
            if t.strip():
                obs_train_vocab.add(normalize_obs_term(t))

    results = {}
    example_dumps = {}
    for field, vocab, normalize_fn, pretty in [
        ("unit", unit_train_vocab, normalize_unit_term, "MeasurementUnit"),
        ("obs", obs_train_vocab, normalize_obs_term, "ScientificObservable"),
    ]:
        buckets = bucket_examples(test_ex, vocab, field, normalize_fn)
        bucket_f1s = defaultdict(list)
        bucket_word_f1s = defaultdict(list)
        bucket_examples_out = defaultdict(list)
        for i, bucket in buckets.items():
            pred_val = parsed_preds[i]["unit"] if field == "unit" else parsed_preds[i]["obs"]
            gold_val = test_ex[i][field]
            f1 = set_f1(pred_val, gold_val, normalize_fn)
            wf1 = word_level_f1(pred_val, gold_val)
            bucket_f1s[bucket].append(f1)
            bucket_word_f1s[bucket].append(wf1)
            bucket_examples_out[bucket].append({
                "caption": test_ex[i]["text"][:150],
                "gold": gold_val,
                "predicted": pred_val,
                "strict_set_f1": round(f1, 3),
                "lenient_word_f1": round(wf1, 3),
            })

        summary = {}
        for b in ["fully_seen", "partially_seen", "fully_unseen"]:
            vals = bucket_f1s.get(b, [])
            wvals = bucket_word_f1s.get(b, [])
            summary[b] = {
                "n": len(vals),
                "mean_strict_f1": float(np.mean(vals)) if vals else None,
                "mean_lenient_word_f1": float(np.mean(wvals)) if wvals else None,
            }
        results[pretty] = summary
        example_dumps[pretty] = {k: v for k, v in bucket_examples_out.items()}
        print(f"\n{pretty}:")
        for b, s in summary.items():
            strict_str = f"{s['mean_strict_f1']:.3f}" if s["mean_strict_f1"] is not None else "n/a"
            lenient_str = f"{s['mean_lenient_word_f1']:.3f}" if s["mean_lenient_word_f1"] is not None else "n/a"
            print(f"  {b}: n={s['n']}, strict F1={strict_str}, lenient word-F1={lenient_str}")

    with open(output_folder / "seen_vs_unseen_examples.json", "w") as f:
        json.dump(example_dumps, f, indent=2)
    print(f"\nWrote full prediction-vs-gold pairs (all buckets) to seen_vs_unseen_examples.json")
    print("Open this file and look at the 'fully_seen' entries -- check whether low-F1")
    print("examples there are genuinely wrong, or just paraphrased correctly.")

    with open(output_folder / "seen_vs_unseen_breakdown.json", "w") as f:
        json.dump(results, f, indent=2)

    # --- plot: 2 subplots (MeasurementUnit, ScientificObservable), 3 buckets each,
    #     strict vs lenient side by side per bucket ---
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    bucket_labels = ["Fully seen\nin training", "Partially seen", "Never seen\nin training"]
    bucket_keys = ["fully_seen", "partially_seen", "fully_unseen"]
    width = 0.35

    for ax, (field_name, summary) in zip(axes, results.items()):
        strict_means = [summary[k]["mean_strict_f1"] if summary[k]["mean_strict_f1"] is not None else 0 for k in bucket_keys]
        lenient_means = [summary[k]["mean_lenient_word_f1"] if summary[k]["mean_lenient_word_f1"] is not None else 0 for k in bucket_keys]
        ns = [summary[k]["n"] for k in bucket_keys]
        x = np.arange(len(bucket_keys))

        bars1 = ax.bar(x - width/2, strict_means, width, label="Strict (exact phrase)", color=PALETTE[2])
        bars2 = ax.bar(x + width/2, lenient_means, width, label="Lenient (word overlap)", color=PALETTE[1])
        for b, m in zip(bars1, strict_means):
            ax.text(b.get_x() + b.get_width()/2, m + 0.02, f"{m:.2f}", ha="center", fontsize=9)
        for b, m, n in zip(bars2, lenient_means, ns):
            ax.text(b.get_x() + b.get_width()/2, m + 0.02, f"{m:.2f}\n(n={n})", ha="center", fontsize=9)

        ax.set_xticks(x); ax.set_xticklabels(bucket_labels)
        ax.set_ylim(0, 1.25); ax.set_ylabel("Mean F1")
        ax.set_title(field_name)
        ax.legend(loc="upper right", fontsize=9, frameon=False)

    fig.suptitle("Model performance vs. training vocabulary coverage\n(strict exact-match vs. lenient word-overlap scoring)", fontsize=14)
    savefig(fig, plots_dir / "seen_vs_unseen_breakdown.png")

    print(f"\nDone. Breakdown saved to seen_vs_unseen_breakdown.json and .png")


if __name__ == "__main__":
    main()

Writing seen_vs_unseen_breakdown.py


In [5]:
!python "seen_vs_unseen_breakdown.py" "$OUT_DIR" --data_file "$DATA_FILE"

Reproducing the exact train/val/test split used during training...
train=535 val=68 test=69
Loading model from /kaggle/working/generative_model_final onto cuda...
Loading weights: 100%|█| 558/558 [00:00<00:00, 1648.79it/s, Materializing param=
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Generating predictions for 69 test captions...
  8/69
  16/69
  24/69
  32/69
  40/69
  48/69
  56/69
  64/69
  69/69

MeasurementUnit:
  fully_seen: n=30, strict F1=0.628, lenient word-F1=0.856
  partially_seen: n=2, strict F1=0.000, lenient word-F1=1.000
  fully_unseen: n=2, strict F1=0.000, lenient word-F1=0.500

ScientificObservable:
  fully_seen: n=19, strict F1=0.689, lenient word-F1=0.797
  partially_seen: n=25, strict F1=0.369, lenient word-F1=0.710
  fully_unseen: n=18, strict F1=0.046, le